In [1]:
!pip install pandas geopy


   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]



# Collect and geocode BMC's official flood hotspots

In [15]:
import pandas as pd

flood_spots = [
    ("Juhu Tara Road and S.V. Road Junction", "Santacruz"),
    ("Milan Subway", "Santacruz"),
    ("Andheri Subway", "Andheri"),
    ("Marol Market, Andheri Kurla Road", "Andheri"),
    ("Vidyanidhi Complex, JVPD North", "Vile Parle"),
    ("Gazdar Bundh", "Santacruz"),
    ("Bhogale Chowk", "Vile Parle"),
    ("M.G. Road and Subhash Road Junction", "Vile Parle"),
    ("Veera Desai Road, Last Bus Stop", "Andheri"),
    ("Kotkar Nalla", "Goregaon"),
    ("Piramal Nagar Nalla", "Goregaon"),
    ("Rani Sati Marg, Makrani Pada", "Malad"),
    ("Valnai Hutment Colonies", "Kandivali"),
    ("Link Road, Guria Pada", "Malad"),
    ("Malad Subway", "Malad"),
    ("Opposite Kandivali Fire Station, S.V. Road", "Kandivali"),
    ("Sambhaji Nagar", "Dahisar"),
    ("Dahisar Subway", "Dahisar")
]

df = pd.DataFrame(
    flood_spots,
    columns=["spot_name", "zone_name"]
)

df["latitude"] = None
df["longitude"] = None
df["coordinate_status"] = "Not geocoded"

df.to_csv("mumbai_flood_hotspots.csv", index=False)

print("Records:", len(df))
print("Exact hotspot coordinates:", df["latitude"].notna().sum())

display(df)

Records: 18
Exact hotspot coordinates: 0


,spot_name,zone_name,latitude,longitude,coordinate_status
0,Juhu Tara Road and S.V. Road Junction,Santacruz,None,None,Not geocoded
1,Milan Subway,Santacruz,None,None,Not geocoded
2,Andheri Subway,Andheri,None,None,Not geocoded
3,"Marol Market, Andheri Kurla Road",Andheri,None,None,Not geocoded
4,"Vidyanidhi Complex, JVPD North",Vile Parle,None,None,Not geocoded
5,Gazdar Bundh,Santacruz,None,None,Not geocoded
6,Bhogale Chowk,Vile Parle,None,None,Not geocoded
7,M.G. Road and Subhash Road Junction,Vile Parle,None,None,Not geocoded
8,"Veera Desai Road, Last Bus Stop",Andheri,None,None,Not geocoded
9,Kotkar Nalla,Goregaon,None,None,Not geocoded


# Retrieve BMC hotspot records

In [16]:
import requests
import pandas as pd

# BMC Disaster Management API
API_URL = "https://dmwebtwo.mcgm.gov.in/api/floodSpot/loadAll"

response = requests.post(
    API_URL,
    json={},
    headers={
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0"
    },
    timeout=60
)

response.raise_for_status()

payload = response.json()

# Extract records
if isinstance(payload, dict):
    spots = payload.get("data") or payload.get("result") or []
else:
    spots = payload

if not isinstance(spots, list):
    raise ValueError("Unexpected API response format")

# Convert to DataFrame
bmc_spots_df = pd.json_normalize(spots)

print("Total records retrieved:", len(bmc_spots_df))
print("\nAvailable columns:")
print(bmc_spots_df.columns.tolist())

# Check important fields
required_columns = [
    "floodingSpotName",
    "floodingSpotLocation",
    "wardName",
    "floodingSpotType",
    "lat",
    "longC",
    "featureId",
    "floodingSpotAWSId"
]

print("\nMissing expected columns:")
print([
    col for col in required_columns
    if col not in bmc_spots_df.columns
])

# Display available records
display(
    bmc_spots_df[
        [col for col in required_columns
         if col in bmc_spots_df.columns]
    ].head(15)
)

Total records retrieved: 110

Available columns:
['fid', 'floodingSpotName', 'floodingSpotType', 'floodingSpotLocation', 'featureId', 'floodingSpotAWSId', 'stretch', 'depth', 'wardName', 'nearestFid', 'nearestDist', 'lat', 'longC', 'weatherStationFID', 'weatherStationFeatureId', 'weatherStationAWSId', 'weatherStationWardName', 'weatherStationName', 'weatherStationLocation']

Missing expected columns:
[]


,floodingSpotName,floodingSpotLocation,wardName,floodingSpotType,lat,longC,featureId,floodingSpotAWSId
0,Jijamata Nagar,"Bander Pakhadi Road,Opp ICICI Bank,Khandiwali ...",R/S,Chronic Flooding Spot,19.2104365001137,72.8309731000217,FS23000000002,WS17350400052
1,Maharastar Nagar,"Link Road,Near Arihant Tower,Kandiwali West,Mu...",R/S,Chronic Flooding Spot,19.2033962001884,72.8395125000085,FS23000000004,WS17350400052
2,Maruti Nagar,"Mahatma Gandhi Road,Near Sulabh Sauchalaya,Kha...",R/S,Chronic Flooding Spot,19.207431000125,72.830320000004,FS23000000001,WS17350400052
3,Shastri Nagar,"Sonawala Cross Lane,Near Shastri Nagar Ground,...",P/S,Chronic Flooding Spot,19.1601619000822,72.8373199000001,FS13000000004,WS16304600047
4,Siddhartha Hospital,"Road No 16,Near Siddhartha Hospital,Goregoan W...",P/S,Chronic Flooding Spot,19.1671826000626,72.8381852000428,FS13000000003,WS16304600047
5,"Hanuman Nagar, Sanjivani Chawl","Vadarpada Road, Near Sree Coaching Classes, Ka...",R/S,Chronic Flooding Spot,19.1963271000621,72.8669399000108,FS23000000008,WS18364300053
6,Air India Colony N Building (Air India Road),"Air India Colony N Building,Air India Road,Nea...",H/E,Flooding Spot,19.0803895999996,72.8727457998286,FS21000000002,WS19207400037
7,Andheri Subway,"Andheri Subway Road, Near Rajdhani Restaurant,...",K/E,Chronic Flooding Spot,19.1243511000103,72.8475674999609,FS11000000016,WS17260000032
8,Milan Subway In Nala,"Milan Subway,Near Milan Subway,Santacruz,Mumba...",H/E,Chronic Flooding Spot,19.0900480001978,72.8430601999571,FS21000000009,WS21000010001
9,"National College, Bandra West","Linking Road, Near National College, Bandra We...",H/W,Chronic Flooding Spot,19.0637486872108,72.8352132619003,FS10000000041,WS15181900028


# Validate and save the raw BMC dataset

In [17]:
# Create a copy of the original API data
bmc_raw_df = bmc_spots_df.copy()

# Convert coordinates to numeric
bmc_raw_df["lat"] = pd.to_numeric(
    bmc_raw_df["lat"], errors="coerce"
)

bmc_raw_df["longC"] = pd.to_numeric(
    bmc_raw_df["longC"], errors="coerce"
)

# Dataset validation
print("Total records:", len(bmc_raw_df))

print("\nMissing values:")
display(bmc_raw_df.isnull().sum().to_frame("Missing values"))

print("\nDuplicate FIDs:", bmc_raw_df["fid"].duplicated().sum())

print("\nRecords with valid coordinates:",
      bmc_raw_df[["lat", "longC"]].notna().all(axis=1).sum())

print("\nFlood spot types:")
display(bmc_raw_df["floodingSpotType"].value_counts())

print("\nWard distribution:")
display(bmc_raw_df["wardName"].value_counts())

# Save original API records
bmc_raw_df.to_csv(
    "mumbai_bmc_flood_hotspots_raw.csv",
    index=False
)

print("\nRaw BMC hotspot dataset saved successfully.")

Total records: 110

Missing values:


,Missing values
fid,0
floodingSpotName,0
floodingSpotType,0
floodingSpotLocation,0
featureId,0
floodingSpotAWSId,0
stretch,0
depth,0
wardName,0
nearestFid,0



Duplicate FIDs: 0

Records with valid coordinates: 110

Flood spot types:


floodingSpotType
Chronic Flooding Spot    98
Flooding Spot            12
Name: count, dtype: int64


Ward distribution:


wardName
P/N    12
F/N    12
K/W    11
H/W    10
R/S     8
H/E     8
P/S     8
K/E     6
R/C     6
R/N     5
M/W     5
N       3
E       3
F/S     3
G/N     2
B       2
T       1
C       1
G/S     1
D       1
L       1
M/E     1
Name: count, dtype: int64


Raw BMC hotspot dataset saved successfully.


# Identify the nine localities

In [18]:
import re

# Locality names and common spelling variations
locality_aliases = {
    "Bandra": ["Bandra"],
    "Santacruz": ["Santacruz", "Santa Cruz", "Vakola"],
    "Vile Parle": ["Vile Parle"],
    "Andheri": ["Andheri"],
    "Jogeshwari": ["Jogeshwari"],
    "Goregaon": ["Goregaon"],
    "Malad": ["Malad"],
    "Kandivali": ["Kandivali", "Kandiwali", "Khandiwali"],
    "Borivali": ["Borivali"]
}

# Search hotspot name and address
bmc_raw_df["search_text"] = (
    bmc_raw_df["floodingSpotName"].fillna("") + " " +
    bmc_raw_df["floodingSpotLocation"].fillna("")
).str.lower()

def identify_localities(text):
    matches = []

    for locality, aliases in locality_aliases.items():
        for alias in aliases:
            pattern = r"\b" + re.escape(alias.lower()) + r"\b"

            if re.search(pattern, text):
                matches.append(locality)
                break

    return matches

# Identify locality matches
bmc_raw_df["matched_localities"] = (
    bmc_raw_df["search_text"].apply(identify_localities)
)

# Count matched and unmatched records
matched_spots = bmc_raw_df[
    bmc_raw_df["matched_localities"].apply(len) > 0
].copy()

unmatched_spots = bmc_raw_df[
    bmc_raw_df["matched_localities"].apply(len) == 0
].copy()

print("Total BMC hotspots:", len(bmc_raw_df))
print("Matched hotspots:", len(matched_spots))
print("Unmatched hotspots:", len(unmatched_spots))

# Locality-wise distribution
locality_counts = (
    matched_spots.explode("matched_localities")
    ["matched_localities"]
    .value_counts()
    .reindex(locality_aliases.keys(), fill_value=0)
)

print("\nHotspots by locality:")
display(locality_counts.to_frame("hotspot_count"))

# Inspect matched records
display(
    matched_spots[
        [
            "floodingSpotName",
            "floodingSpotLocation",
            "wardName",
            "lat",
            "longC",
            "matched_localities"
        ]
    ]
)

# Inspect unmatched records for later review
print("\nUnmatched hotspot records:")
display(
    unmatched_spots[
        [
            "floodingSpotName",
            "floodingSpotLocation",
            "wardName",
            "lat",
            "longC"
        ]
    ]
)

Total BMC hotspots: 110
Matched hotspots: 53
Unmatched hotspots: 57

Hotspots by locality:


,hotspot_count
matched_localities,
Bandra,6
Santacruz,5
Vile Parle,0
Andheri,10
Jogeshwari,3
Goregaon,5
Malad,12
Kandivali,6
Borivali,6


,floodingSpotName,floodingSpotLocation,wardName,lat,longC,matched_localities
0,Jijamata Nagar,"Bander Pakhadi Road,Opp ICICI Bank,Khandiwali ...",R/S,19.210437,72.830973,[Kandivali]
1,Maharastar Nagar,"Link Road,Near Arihant Tower,Kandiwali West,Mu...",R/S,19.203396,72.839513,[Kandivali]
2,Maruti Nagar,"Mahatma Gandhi Road,Near Sulabh Sauchalaya,Kha...",R/S,19.207431,72.830320,[Kandivali]
5,"Hanuman Nagar, Sanjivani Chawl","Vadarpada Road, Near Sree Coaching Classes, Ka...",R/S,19.196327,72.866940,[Kandivali]
7,Andheri Subway,"Andheri Subway Road, Near Rajdhani Restaurant,...",K/E,19.124351,72.847567,[Andheri]
8,Milan Subway In Nala,"Milan Subway,Near Milan Subway,Santacruz,Mumba...",H/E,19.090048,72.843060,[Santacruz]
9,"National College, Bandra West","Linking Road, Near National College, Bandra We...",H/W,19.063749,72.835213,[Bandra]
11,"Akurli Road, Near Thakur House, Kandivali (E)","Akurli Road, Near Thakur House, Kandivali (E)",R/S,19.201972,72.856240,[Kandivali]
14,"Dahanukar Wadi Cemetry Road, Kandivali (W)","Dahanukar Wadi Cemetry Road, Kandivali (W)",R/S,19.206774,72.837131,[Kandivali]
15,"Abhinav Nagar, Shri krishna Nagar, National Park","Shri krishna Nagar Road, Near Fab The Gym, Bor...",R/C,19.232352,72.865146,[Borivali]



Unmatched hotspot records:


,floodingSpotName,floodingSpotLocation,wardName,lat,longC
3,Shastri Nagar,"Sonawala Cross Lane,Near Shastri Nagar Ground,...",P/S,19.160162,72.837320
4,Siddhartha Hospital,"Road No 16,Near Siddhartha Hospital,Goregoan W...",P/S,19.167183,72.838185
6,Air India Colony N Building (Air India Road),"Air India Colony N Building,Air India Road,Nea...",H/E,19.080390,72.872746
10,Pratiksha Nagar,"Saradar Nagar Road no 1,Near Railway East-West...",F/N,19.040189,72.866710
12,MG Road upto Hindusthan Naka,MG Road upto Hindusthan Naka,R/S,19.208776,72.829687
13,"Ganesh Nagar , 60 ft DP Road","Ganesh Nagar , 60 ft DP Road",R/S,19.203723,72.829357
17,Nana Chowk,"Nana Chowk Road,Near Municipal Building,Sandhu...",D,18.961939,72.813155
20,IC Colony (Pearl Apartment),"Road No 4,Inside Pearl Aparatment,Dahisar West...",R/N,19.246667,72.846355
22,"Naik Nagar, Kali Bharni,Qadri Masjid","Naik Nagar, Kali Bharni, Ward no. 183, Qadri M...",G/N,19.051176,72.864477
24,"Mhanaji Rajuji Chawl no, 2, Room no, 13, Sitar...","Sitaram Jadhav Marg, Near Sujata Lady Fashion,...",G/S,18.996375,72.827661


# Inspect possible Vile Parle hotspot

In [19]:
# Inspect unmatched hotspots around the Vile Parle area

vile_parle_candidates = unmatched_spots[
    (unmatched_spots["lat"].between(19.09, 19.12)) &
    (unmatched_spots["longC"].between(72.82, 72.88))
].copy()

print("Potential Vile Parle area candidates:",
      len(vile_parle_candidates))

display(
    vile_parle_candidates[
        [
            "floodingSpotName",
            "floodingSpotLocation",
            "wardName",
            "lat",
            "longC"
        ]
    ].sort_values("lat")
)

Potential Vile Parle area candidates: 3


,floodingSpotName,floodingSpotLocation,wardName,lat,longC
86,Irla Masjid & Juhu Junction,Irla Masjid & Juhu Junction,K/W,19.111512,72.837206
92,Juhu Gully,Juhu Gully,K/W,19.114326,72.839419
93,"Irla Jn,S.V. Road Ajit Glass Jn","Irla Jn,S.V. Road Ajit Glass Jn",K/W,19.114955,72.842685


# Add reviewed classifications

In [20]:
# Manually review the three candidate records

reviewed_localities = {
    "Irla Masjid & Juhu Junction": ("Vile Parle", "Medium"),
    "Juhu Gully": ("Uncertain - Juhu/Vile Parle border", "Low"),
    "Irla Jn,S.V. Road Ajit Glass Jn": ("Vile Parle", "Medium")
}

for spot_name, (locality, confidence) in reviewed_localities.items():
    mask = bmc_raw_df["floodingSpotName"] == spot_name

    bmc_raw_df.loc[mask, "reviewed_locality"] = locality
    bmc_raw_df.loc[mask, "match_confidence"] = confidence

display(
    bmc_raw_df[
        bmc_raw_df["floodingSpotName"].isin(reviewed_localities)
    ][[
        "floodingSpotName",
        "floodingSpotLocation",
        "reviewed_locality",
        "match_confidence"
    ]]
)

,floodingSpotName,floodingSpotLocation,reviewed_locality,match_confidence
86,Irla Masjid & Juhu Junction,Irla Masjid & Juhu Junction,Vile Parle,Medium
92,Juhu Gully,Juhu Gully,Uncertain - Juhu/Vile Parle border,Low
93,"Irla Jn,S.V. Road Ajit Glass Jn","Irla Jn,S.V. Road Ajit Glass Jn",Vile Parle,Medium


# Create a consolidated locality assignment

In [21]:
import pandas as pd

def assign_locality(row):
    # Give priority to manual review
    reviewed = row.get("reviewed_locality")

    if pd.notna(reviewed):
        if str(reviewed).startswith("Uncertain"):
            return "Needs Review"
        return reviewed

    # Use preliminary text matching
    matches = row["matched_localities"]

    if len(matches) == 1:
        return matches[0]
    elif len(matches) > 1:
        return "Multiple Matches - Review"

    return "Unmatched"


bmc_raw_df["provisional_locality"] = bmc_raw_df.apply(
    assign_locality, axis=1
)

print("Locality assignment summary:")
print(
    bmc_raw_df["provisional_locality"]
    .value_counts(dropna=False)
)

print("\nLocality-wise hotspot counts:")
display(
    bmc_raw_df[
        bmc_raw_df["provisional_locality"].isin([
            "Bandra", "Santacruz", "Vile Parle",
            "Andheri", "Jogeshwari", "Goregaon",
            "Malad", "Kandivali", "Borivali"
        ])
    ]["provisional_locality"]
    .value_counts()
    .reindex([
        "Bandra", "Santacruz", "Vile Parle",
        "Andheri", "Jogeshwari", "Goregaon",
        "Malad", "Kandivali", "Borivali"
    ], fill_value=0)
)

Locality assignment summary:
provisional_locality
Unmatched       54
Malad           12
Andheri         10
Kandivali        6
Bandra           6
Borivali         6
Santacruz        5
Goregaon         5
Jogeshwari       3
Vile Parle       2
Needs Review     1
Name: count, dtype: int64

Locality-wise hotspot counts:


provisional_locality
Bandra         6
Santacruz      5
Vile Parle     2
Andheri       10
Jogeshwari     3
Goregaon       5
Malad         12
Kandivali      6
Borivali       6
Name: count, dtype: int64

# Inspect unmatched records

In [22]:
unmatched_review = bmc_raw_df[
    bmc_raw_df["provisional_locality"] == "Unmatched"
].copy()

display(
    unmatched_review[
        [
            "fid",
            "floodingSpotName",
            "floodingSpotLocation",
            "wardName",
            "lat",
            "longC"
        ]
    ].sort_values(["wardName", "lat"])
)

,fid,floodingSpotName,floodingSpotLocation,wardName,lat,longC
75,75,Masjid Bunder Railway Track in B ward,Masjid Bunder Railway Track in B ward,B,18.952530,72.838199
74,74,Junction of P D ‘Mello Road & S.V.P. Road,Junction of P D ‘Mello Road & S.V.P. Road,B,18.957412,72.842512
59,59,Gol Deval Alankar Cinema Junction,"S V P Road, Near Alankar Cinema, Kumbharwada, ...",C,18.957994,72.825939
17,17,Nana Chowk,"Nana Chowk Road,Near Municipal Building,Sandhu...",D,18.961939,72.813155
60,60,"Gate no 4, Nair Hospital, Murzban Colony","Dr Android Nair Marg, Near Murzban Colony, Mum...",E,18.974304,72.822995
61,61,"Patra Chwal, Ghas Bazar","Sane Guruji Marg, Near Bal Gopal Mitra Mandal,...",E,18.975540,72.821880
99,99,Prabha Sadan Bldg,"Opp Argipada Police station, R B Chadorkar Mar...",E,18.976853,72.824382
33,33,Shakkar panchyayat R.A.Kidwai,"Shakkar panchyayat R.A.Kidwai, Mumbai",F/N,19.011694,72.854410
36,36,Bank of baroda R A Kidwai,"Bank of baroda R.A.Kidwai, Wadala",F/N,19.015917,72.857809
32,32,Wadala railway station,Wadala railway station,F/N,19.016988,72.858870


# Manually review the 7 potential matches

In [23]:
# Additional potential locality assignments based on geographic review

additional_reviews = {
    45: ("Malad", "Medium"),
    43: ("Goregaon", "Medium"),
    3: ("Goregaon", "High"),
    4: ("Goregaon", "High"),
    47: ("Borivali", "Medium"),
    13: ("Kandivali", "Medium"),
    12: ("Needs Review - Kandivali/Malad", "Low")
}

for fid, (locality, confidence) in additional_reviews.items():

    mask = bmc_raw_df["fid"] == fid

    bmc_raw_df.loc[mask, "reviewed_locality"] = locality
    bmc_raw_df.loc[mask, "match_confidence"] = confidence

    if locality.startswith("Needs Review"):
        bmc_raw_df.loc[mask, "provisional_locality"] = "Needs Review"
    else:
        bmc_raw_df.loc[mask, "provisional_locality"] = locality

display(
    bmc_raw_df[
        bmc_raw_df["fid"].isin(additional_reviews)
    ][[
        "fid",
        "floodingSpotName",
        "provisional_locality",
        "match_confidence"
    ]].sort_values("fid")
)

,fid,floodingSpotName,provisional_locality,match_confidence
3,3,Shastri Nagar,Goregaon,High
4,4,Siddhartha Hospital,Goregaon,High
12,12,MG Road upto Hindusthan Naka,Needs Review,Low
13,13,"Ganesh Nagar , 60 ft DP Road",Kandivali,Medium
43,43,Aarey Road,Goregaon,Medium
45,45,Mamu Gully,Malad,Medium
47,47,"Near Gagangiri Bldg., D D Borge Road",Borivali,Medium


# Verify the complete classification

In [24]:
# Verify all 110 BMC records

print("Total records:", len(bmc_raw_df))
print("\nClassification summary:")

print(
    bmc_raw_df["provisional_locality"]
    .value_counts(dropna=False)
)

print("\nNine-locality hotspot counts:")

study_localities = [
    "Bandra", "Santacruz", "Vile Parle",
    "Andheri", "Jogeshwari", "Goregaon",
    "Malad", "Kandivali", "Borivali"
]

locality_counts = (
    bmc_raw_df[
        bmc_raw_df["provisional_locality"].isin(study_localities)
    ]["provisional_locality"]
    .value_counts()
    .reindex(study_localities, fill_value=0)
)

display(locality_counts.to_frame("hotspot_count"))

print("\nDuplicate FIDs:", bmc_raw_df["fid"].duplicated().sum())
print("Missing coordinates:", bmc_raw_df[["lat", "longC"]].isna().any(axis=1).sum())

Total records: 110

Classification summary:
provisional_locality
Unmatched       47
Malad           13
Andheri         10
Goregaon         8
Kandivali        7
Borivali         7
Bandra           6
Santacruz        5
Jogeshwari       3
Needs Review     2
Vile Parle       2
Name: count, dtype: int64

Nine-locality hotspot counts:


,hotspot_count
provisional_locality,
Bandra,6
Santacruz,5
Vile Parle,2
Andheri,10
Jogeshwari,3
Goregaon,8
Malad,13
Kandivali,7
Borivali,7



Duplicate FIDs: 0
Missing coordinates: 0


# Save the consolidated BMC dataset

In [25]:
# Save the consolidated BMC flood hotspot dataset

output_file = "mumbai_bmc_flood_hotspots_locality_matched.csv"

bmc_raw_df.to_csv(output_file, index=False)

print("Dataset saved successfully:", output_file)
print("Total records exported:", len(bmc_raw_df))

# Verify the exported file
verified_df = pd.read_csv(output_file)

print("Verified records:", len(verified_df))
print("Verified columns:", len(verified_df.columns))

Dataset saved successfully: mumbai_bmc_flood_hotspots_locality_matched.csv
Total records exported: 110
Verified records: 110
Verified columns: 24


# Inspect BMC hotspot data

In [26]:
import pandas as pd

# Load locality-matched BMC hotspot dataset
bmc_df = pd.read_csv("mumbai_bmc_flood_hotspots_locality_matched.csv")

print("Dataset shape:", bmc_df.shape)

print("\nLocality classification:")
display(bmc_df["provisional_locality"].value_counts(dropna=False))

print("\nMatching confidence:")
display(bmc_df["match_confidence"].value_counts(dropna=False))

print("\nHotspot type distribution:")
display(bmc_df["floodingSpotType"].value_counts(dropna=False))

print("\nSample records:")
display(
    bmc_df[
        ["fid", "floodingSpotName", "provisional_locality",
         "match_confidence", "floodingSpotType"]
    ].head(10)
)

Dataset shape: (110, 24)

Locality classification:


provisional_locality
Unmatched       47
Malad           13
Andheri         10
Goregaon         8
Kandivali        7
Borivali         7
Bandra           6
Santacruz        5
Jogeshwari       3
Needs Review     2
Vile Parle       2
Name: count, dtype: int64


Matching confidence:


match_confidence
NaN       100
Medium      6
High        2
Low         2
Name: count, dtype: int64


Hotspot type distribution:


floodingSpotType
Chronic Flooding Spot    98
Flooding Spot            12
Name: count, dtype: int64


Sample records:


,fid,floodingSpotName,provisional_locality,match_confidence,floodingSpotType
0,0,Jijamata Nagar,Kandivali,NaN,Chronic Flooding Spot
1,1,Maharastar Nagar,Kandivali,NaN,Chronic Flooding Spot
2,2,Maruti Nagar,Kandivali,NaN,Chronic Flooding Spot
3,3,Shastri Nagar,Goregaon,High,Chronic Flooding Spot
4,4,Siddhartha Hospital,Goregaon,High,Chronic Flooding Spot
5,5,"Hanuman Nagar, Sanjivani Chawl",Kandivali,NaN,Chronic Flooding Spot
6,6,Air India Colony N Building (Air India Road),Unmatched,NaN,Flooding Spot
7,7,Andheri Subway,Andheri,NaN,Chronic Flooding Spot
8,8,Milan Subway In Nala,Santacruz,NaN,Chronic Flooding Spot
9,9,"National College, Bandra West",Bandra,NaN,Chronic Flooding Spot


# Calculate hotspot counts for each locality

In [27]:
# Nine study localities
localities = [
    "Bandra", "Santacruz", "Vile Parle",
    "Andheri", "Jogeshwari", "Goregaon",
    "Malad", "Kandivali", "Borivali"
]

# Keep only records assigned to the nine localities
matched_bmc_df = bmc_df[
    bmc_df["provisional_locality"].isin(localities)
].copy()

# Count hotspot types by locality
hotspot_summary = (
    matched_bmc_df
    .pivot_table(
        index="provisional_locality",
        columns="floodingSpotType",
        values="fid",
        aggfunc="count",
        fill_value=0
    )
    .reindex(localities, fill_value=0)
    .reset_index()
)

# Rename columns
hotspot_summary.columns.name = None
hotspot_summary = hotspot_summary.rename(columns={
    "provisional_locality": "zone_name",
    "Chronic Flooding Spot": "chronic_flooding_spot_count",
    "Flooding Spot": "flooding_spot_count"
})

# Calculate total hotspot count
hotspot_summary["total_hotspot_count"] = (
    hotspot_summary["chronic_flooding_spot_count"]
    + hotspot_summary["flooding_spot_count"]
)

print("Locality-wise BMC hotspot summary:")
display(hotspot_summary)

print("\nTotal assigned hotspots:",
      hotspot_summary["total_hotspot_count"].sum())

print("Unmatched records:",
      (bmc_df["provisional_locality"] == "Unmatched").sum())

print("Records requiring review:",
      (bmc_df["provisional_locality"] == "Needs Review").sum())

Locality-wise BMC hotspot summary:


,zone_name,chronic_flooding_spot_count,flooding_spot_count,total_hotspot_count
0,Bandra,4,2,6
1,Santacruz,4,1,5
2,Vile Parle,2,0,2
3,Andheri,9,1,10
4,Jogeshwari,3,0,3
5,Goregaon,8,0,8
6,Malad,10,3,13
7,Kandivali,7,0,7
8,Borivali,6,1,7



Total assigned hotspots: 61
Unmatched records: 47
Records requiring review: 2


# Save the locality-wise hotspot summary

In [28]:
# Save locality-wise hotspot summary
output_file = "mumbai_locality_hotspot_summary.csv"

hotspot_summary.to_csv(output_file, index=False)

# Verify saved file
verified_hotspots = pd.read_csv(output_file)

print("Saved file:", output_file)
print("Verified shape:", verified_hotspots.shape)
print("Missing values:", verified_hotspots.isnull().sum().sum())

display(verified_hotspots)

Saved file: mumbai_locality_hotspot_summary.csv
Verified shape: (9, 4)
Missing values: 0


,zone_name,chronic_flooding_spot_count,flooding_spot_count,total_hotspot_count
0,Bandra,4,2,6
1,Santacruz,4,1,5
2,Vile Parle,2,0,2
3,Andheri,9,1,10
4,Jogeshwari,3,0,3
5,Goregaon,8,0,8
6,Malad,10,3,13
7,Kandivali,7,0,7
8,Borivali,6,1,7
